# Linear Agent — user perspective

**What it does.** Wraps every `mcp__linear__*` call behind a Pydantic-validated contract (`LinearOutput`). All other agents go through this one — there is no other Linear write path.

**Entry points.**
- `await run_linear_agent(prompt)` — raw async call, returns text or `None`.
- `await run_validated_linear_agent(operation, payload)` — wraps the above with retry + Pydantic validation. **Use this one** in every operational caller.

**G5 write guard.** Write operations dispatch through a stack-inspecting decorator (`linear_write_guard`) that denies any frame originating from `risk_agent.py` outside the operator-delegated `approve_improvement_trigger` path (RISK-04 layer 4).

**Cost.** Live runs use Linear MCP. The example below is a `read` (cheap).

In [ ]:
import sys
from pathlib import Path

# Robust _helpers.py discovery — works whether jupyter lab was launched from
# the repo root, notebooks/, or notebooks/agents/. The loop walks up from cwd
# until it finds the parent that holds _helpers.py and inserts it on sys.path.
nb_dir = Path.cwd()
for _parent in (nb_dir, *nb_dir.parents):
    if (_parent / "_helpers.py").exists():
        sys.path.insert(0, str(_parent))
        break

from _helpers import ensure_src_on_path, runtime_summary

ROOT = ensure_src_on_path()
print("HSB_RUNTIME_<AGENT> selection:\n" + runtime_summary())

## Construct the inputs

`run_validated_linear_agent` takes two positional args: the operation string (one of the recognised verbs the system prompt knows about — `read`, `create`, `update`, `comment`, `create_subtasks`, `link_pr`) and a payload dict. The agent's system prompt instructs the LLM how to map them onto `mcp__linear__*` tool calls.

In [ ]:
example_operation = "read"
example_payload = {"kind": "teams"}  # ask Linear for the available teams
print(f"operation = {example_operation!r}")
print(f"payload   = {example_payload!r}")

## Live invocation

Gates: `HSB_NOTEBOOK_RUN_LIVE=1`. First-run warning: `mcp-remote` to `mcp.linear.app/mcp` triggers an interactive OAuth flow if you've never authenticated this machine. Run the smoke test in `linear_agent.py` once from the CLI before the notebook, or run it from a terminal where you can complete the browser flow.

In [ ]:
from _helpers import assert_g1_safe, gated, live_mode, selected_runtime

if not live_mode():
    print(gated("Linear Agent live run"))
else:
    assert_g1_safe()
    print(f"(running on HSB_RUNTIME_LINEAR={selected_runtime('linear')!r})")

    from hsb.agents.linear_agent import run_validated_linear_agent

    # Top-level await — IPython already runs the cell inside an active event
    # loop, so asyncio.run() would refuse to nest.
    out = await run_validated_linear_agent(  # noqa: F704
        operation=example_operation,
        payload=example_payload,
    )
    print("operation:", out.operation)
    print("result:   ", out.result)
    print("entities: ", len(out.linear_entities))